# Spark Environment
This notebook includes examples of how to connect and configure Spark to run in a local environment and to manage the dependencies and options required to read/write data to an S3 comptabile object storage.

### Environment Notes
* By convention credentials and secrets are injected into the runtime as environment variables.
  * BASH environment variables are usually named in all caps.
  * Scala variables are generally lowercase.
* AWS/S3 variables are prefixed using `OBJECTS_*`

## Configure Runtime
Jupyter can be configured to work with [BeakerX](http://beakerx.com/), a set of kernels and extensions which provide JVM support, Spark cluster support, and interactive plots/tables/forms.

* `%%classpath`: One of BeakerX's "magics" provides support for dynamically loading dependencies through the use of Maven.

In [ ]:
%%classpath add mvn
org.apache.spark spark-sql_2.11 2.4.5
org.apache.spark spark-kubernetes_2.11 2.4.5
tech.tablesaw tablesaw-core 0.37.3
tech.tablesaw tablesaw-beakerx 0.37.3
tech.tablesaw tablesaw-json 0.37.3
tech.tablesaw tablesaw-jsplot 0.37.3
com.amazonaws aws-java-sdk-s3 1.11.744

Packages:

* Spark SQL: Spark's structured data interface
* Spark Kubernetes: Set of tools and utilities that allows Spark to run workers on Kubernetes.
* [TableSaw](https://github.com/jtablesaw/tablesaw): Flexible dataframe and visualization classes for Java and other JRE languages. (Rough equivalent of Pandas in the Python ecosystem.)
* Amazon AWS Java SDK for Amazon S3

In [ ]:
%classpath add jar /opt/spark/jars/joda-time-2.9.9.jar
%classpath add jar /opt/spark/jars/httpclient-4.5.3.jar
%classpath add jar /opt/spark/jars/aws-java-sdk-s3-1.11.534.jar
%classpath add jar /opt/spark/jars/aws-java-sdk-kms-1.11.534.jar
%classpath add jar /opt/spark/jars/aws-java-sdk-dynamodb-1.11.534.jar
%classpath add jar /opt/spark/jars/aws-java-sdk-core-1.11.534.jar
%classpath add jar /opt/spark/jars/aws-java-sdk-1.11.534.jar
%classpath add jar /opt/spark/jars/hadoop-aws-3.1.2.jar
%classpath add jar /opt/spark/jars/slf4j-api-1.7.29.jar
%classpath add jar /opt/spark/jars/slf4j-log4j12-1.7.29.jar

_JARS included in the commands above are needed to allow Spark to read and write data from MinIO. They are included as part of the Spark dependencies in the container._

In [ ]:
%import tech.tablesaw.api.*
%import tech.tablesaw.columns.*

// Display TableSaw tables with BeakerX table display widget
tablesaw.beakerx.TablesawDisplayer.register()

## Working With Data
* Retrieve files from the Internet
* Read/write data to S3 compatible object storage

In [ ]:
// Example 1: Download URL contents to a String in Scala
import scala.io.Source
import java.net.URL
import tech.tablesaw.api.{Table}
import tech.tablesaw.io.csv.{CsvReadOptions}

// Retrieve remote data and convert the results to a string
val surl1 = "https://oak-tree.tech/documents/96/framingham.csv"
val d1 = Source.fromURL(surl1)
println(d1.mkString)

### Working with Object Storage
_Example adapted from the [MinIO cookbook](https://github.com/minio/cookbook/blob/master/docs/aws-sdk-for-java-with-minio.md)._

#### Upload String Data to S3 Object

In [ ]:
// Imports required to work with Buckets
import com.amazonaws.ClientConfiguration
import com.amazonaws.auth.{BasicAWSCredentials,AWSStaticCredentialsProvider}
import com.amazonaws.services.s3.AmazonS3Client
import com.amazonaws.client.builder.AwsClientBuilder

import com.amazonaws.regions.Regions
import com.amazonaws.services.s3.{AmazonS3,AmazonS3ClientBuilder}
import com.amazonaws.services.s3.model.Bucket
import com.amazonaws.services.s3.model.{
    GetObjectRequest,PutObjectRequest, S3Object}


// Retrieve credentials from environment
val OBJECT_STORAGE_URL = sys.env.getOrElse("OBJECTS_ENDPOINT", "")
val OBJECT_STORAGE_ACCESSID = sys.env.getOrElse("OBJECTS_ACCESSID", "")
val OBJECT_STORAGE_SECRET = sys.env.getOrElse("OBJECTS_SECRET", "")
val OBJECTS_PLAYGROUND_BUCKET = "playground"
val OBJECTS_PREFIX = sys.env.getOrElse("HOSTNAME", "")


// S3 Access Credentials and Client Configuration
println(s"Object Storage URL: $OBJECT_STORAGE_URL")

// Client Configfuration
val awsconf = new ClientConfiguration()
awsconf.setSignerOverride("AWSS3V4SignerType")

// Credentials
val awsauth = new BasicAWSCredentials(
    OBJECT_STORAGE_ACCESSID, OBJECT_STORAGE_SECRET)

In [ ]:
// Initialize S3 Client Instance
val s3 = AmazonS3ClientBuilder
    .standard()
    .withEndpointConfiguration(
        new AwsClientBuilder.EndpointConfiguration(
            OBJECT_STORAGE_URL, Regions.US_EAST_1.name()))
    .withPathStyleAccessEnabled(true)
    .withClientConfiguration(awsconf)
    .withCredentials(new AWSStaticCredentialsProvider(awsauth))
    .build()

In [ ]:
// The "key" name is created from the playground bucket and 

// container hostname to prevent collissions on shared infrastructure.
var fkey = s"$OBJECTS_PREFIX/framingham.csv"
println(s"File key: $fkey")

// Upload the data from the CSV retrieved earlier to the bucket.
s3.putObject(OBJECTS_PLAYGROUND_BUCKET, fkey, d1.mkString)

### Working With Structured Data

In [ ]:
// Example 2: Retrieve CSV file and initialize TableSaw DataFrame
val durl = new URL(surl)
val options1 = CsvReadOptions.builder(durl).header(true)

// Retrieve data and visualize the structure/schema of the dataframe
val dt1 = Table.read().usingOptions(options1)
dt1.structure()

In [ ]:
// View Framingham Data
dt1

## Exercise: Stage DOT Flight Data to MinIO
Prior to being able to analyze data in Spark, if must be staged to a specific location that is accessible to the workers (executors) which will process it. Often this will be to a data lake or storage cluster (such as S3 object storage or HDFS). 

The following files will be utilized throughout the remainder of the course. Write a simple program to download the data and stage it within the MinIO cluster in this enviornment so that Spark is able to consume it from the provided bucket/file keys.

In [ ]:
val FLIGHTS_DATAFILES = Map(
    "web-age/wa2843/airports.json" -> "https://oak-tree.tech/documents/103/airports.json",
    "web-age/wa2843/dot.flights.2018.json" -> "https://oak-tree.tech/documents/105/dot.flights.2018.json",
    "web-age/wa2843/dot.flights.2017-0102.json" -> "https://oak-tree.tech/documents/104/dot.flights.2017-0102.json"
)

// Reminder/Hint: Iteration over map
for ((k,v) <- FLIGHTS_DATAFILES) println(s"File key: $k\nSource URL: $v\n")